# Silver — CRM Sales Details
Sales transactions from the CRM.

`bronze.crm_sales_details` → `silver.crm_sales`

## Init

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType, DateType

CATALOG = "workspace"

## Read bronze table

In [ ]:
df = spark.table(f"{CATALOG}.bronze.crm_sales_details")

## Transformations

### Trim all string columns

In [ ]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Clean dates
Dates arrive as integers like `20101229`. Zero or wrong-length values are invalid → `NULL`; the rest are parsed as `yyyyMMdd`.

In [ ]:
def clean_date(c):
    return (
        F.when((col(c) == 0) | (F.length(col(c)) != 8), None)
         .otherwise(F.to_date(col(c).cast("string"), "yyyyMMdd"))
    )

df = (
    df
    .withColumn("sls_order_dt", clean_date("sls_order_dt"))
    .withColumn("sls_ship_dt",  clean_date("sls_ship_dt"))
    .withColumn("sls_due_dt",   clean_date("sls_due_dt"))
)

### Fix invalid prices
If price is missing or non-positive, derive it from `sales / quantity`.

In [ ]:
df = df.withColumn(
    "sls_price",
    F.when(
        col("sls_price").isNull() | (col("sls_price") <= 0),
        F.when(col("sls_quantity") != 0, col("sls_sales") / col("sls_quantity")).otherwise(None)
    ).otherwise(col("sls_price"))
)

### Rename to business-friendly names

In [ ]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity check

In [ ]:
df.limit(10).display()

## Write silver table

In [ ]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{CATALOG}.silver.crm_sales")

In [ ]:
%sql
SELECT * FROM workspace.silver.crm_sales LIMIT 10;